# 03 · MAPPO và MADDPG trên mạng nhỏ (so sánh có phạm vi hạn chế)

Hai thuật toán được **cài đặt từ đầu bằng numpy** (`marl_baselines/algos_np.py`), *không* phải cài đặt chuẩn của thư viện; kết quả chỉ áp dụng cho các mạng nhỏ ($d\le27$), 10 hạt giống,
huấn luyện $10^5$ bước trên mạng chưa bị sốc. Phần đầu notebook đọc bảng kết quả đã lưu; phần sau huấn luyện lại **một** tác nhân MAPPO (khoảng vài chục giây).

In [1]:
# --- setup: chạy được cả trên Colab lẫn máy cục bộ ---
import os, sys, subprocess
REPO_URL = "https://github.com/hieuthanhpham12/marl-vi-trade-crisis.git"
if not os.path.isdir("icta2026"):
    if os.path.isdir("../icta2026"):
        os.chdir("..")                                   # đang chạy trong notebooks/
    else:
        subprocess.run(["git", "clone", REPO_URL], check=True)
        os.chdir("marl-vi-trade-crisis")
for sub in ("icta2026", "supplementary", "marl_baselines"):
    sys.path.insert(0, os.path.abspath(sub))
import numpy as np, pandas as pd, json
import matplotlib.pyplot as plt
print("thư mục làm việc:", os.getcwd())

thư mục làm việc: /content/marl-vi-trade-crisis


In [2]:
R = json.load(open("marl_baselines/results/res_main_none.json"))
rows = []
for o in R:
    sp = o["spec"]
    for mode in ("frozen", "online"):
        f = o[mode]; rows.append(dict(mạng="x".join(map(str, sp["net"][1:])), thuật_toán=sp["algo"], chế_độ=mode, seed=sp["seed"],
                                       ok=f["g50_ok"], RT_G50=f["g50_rt"], G_pre=f["G_pre"], G_cuối=f["G_final"], t_G_le_1=f["t_G1"]))
df = pd.DataFrame(rows)
df[df.chế_độ == "frozen"].groupby(["mạng", "thuật_toán"]).agg(Succ=("ok", "mean"), RT_G50=("RT_G50", "mean"), G_pre=("G_pre", "mean"), G_cuối=("G_cuối", "mean")).round(3)

Succ  RT_G50  G_pre  G_cuối
mạng    thuật_toán                             
2x2     MADDPG       1.0     9.0  0.172   2.180
        MAPPO        1.0     9.0  0.086   0.695
3x3     MADDPG       0.9    46.5  1.640   4.014
        MAPPO        1.0     6.5  0.058   0.514
3x3x1x3 MADDPG       1.0     3.4  1.778   4.531
        MAPPO        1.0     3.0  0.088   0.322

In [3]:
import time
from rlrun import train, deploy
import ext
hp = json.load(open("marl_baselines/best_hp.json"))["MAPPO"]
NET, BUDGET = ("single", 2, 2), 100_000
t0 = time.time(); agent, env = train("MAPPO", NET, 0, hp, "none", BUDGET)
res = deploy(agent, "MAPPO", NET, 0, hp, "frozen", 800, traj=True)
print(f"huấn luyện {time.time()-t0:.0f}s | phục hồi theo G50: {res['g50_rt']} bước | G_pre={res['G_pre']:.3f} | G_cuối={res['G_final']:.3f}")

M = ext.schedulers(); G_sr = ext.run_ext(M["SR-PLR-MARL"](), ext.make_single(2, 2), 0, N=800, shocks=((300, "price", 2.0),), sigma=0.01)["G"]
plt.figure(figsize=(6, 3.4))
plt.semilogy(np.maximum(res["G"], 1e-4), label="MAPPO (đóng băng, 1e5 bước huấn luyện)")
plt.semilogy(np.maximum(G_sr, 1e-4), label="SR-PLR-MARL (không huấn luyện)")
plt.axvline(300, color="k", ls=":"); plt.xlabel("tập/bước"); plt.ylabel(r"$G_\lambda$"); plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

huấn luyện 32s | phục hồi theo G50: 9.0 bước | G_pre=0.154 | G_cuối=0.415


**Cách đọc.** Sau huấn luyện MAPPO phục hồi nhanh hơn SR-PLR-MARL nhưng có phần dư cuối cao hơn PG/EG; MADDPG kém ổn định hơn. Ưu thế của SR-PLR-MARL nằm ở việc
*không cần huấn luyện* và có bảo đảm hội tụ hầu chắc chắn sau cú sốc cuối cùng (với giả thiết (A5) và phép chiếu chính xác), không nằm ở tốc độ.